# 7C. Representation Fit Analysis Colab

This notebook checks whether RGB tends to win specifically on videos where pose quality is weaker, which helps determine whether the chosen representation matches the data conditions.


## Reason, Approach, Result Interpretation

**Reason**
- Stage 7 showed mixed pose-vs-RGB results.
- The next question is not only which model is better on average, but *why* it is better on certain videos.

**Approach**
- Join pose sequence quality summaries with pose and RGB predictions at the video level.
- Compare errors on the same valid videos.
- Bucket cases by pose visibility and pose confidence.

**How to interpret the result**
- If RGB wins mostly where pose quality is weak, then RGB is compensating for incomplete pose information.
- If pose still wins even in weak-pose buckets, then the issue is not just missing joints.
- If RGB wins in high-pose-quality buckets, then RGB may be helping with context or semantics rather than visibility alone.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Environment Setup

**Why this section exists**
- The analysis needs one additional utility script plus the Stage 5 and Stage 7 artifacts.

**Approach**
- Resolve the Drive project root.
- Sync the representation-fit analysis script into Drive.
- Resolve the pose and RGB summary paths.

**How to interpret the result**
- If the printed files exist, the notebook is ready to run.


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

FIT_REL = Path('artifacts/3_Modeling/analyze_representation_fit.py')

src = CODE_ROOT / FIT_REL
dst = DRIVE_PROJECT_ROOT / FIT_REL
dst.parent.mkdir(parents=True, exist_ok=True)
if src.exists() and (not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime + 1.0):
    shutil.copy2(src, dst)
    print('[sync] copied analysis script to Drive')
else:
    print('[sync] keeping Drive analysis script')

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
POSE_SUMMARY = ANNOTATION_DIR / 'pose_sequence_summary.csv'
RGB_SUMMARY = ANNOTATION_DIR / 'rgb_feature_summary_selected.csv'
TRAINING_DIR = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs'

print('POSE_SUMMARY =', POSE_SUMMARY, POSE_SUMMARY.exists())
print('RGB_SUMMARY =', RGB_SUMMARY, RGB_SUMMARY.exists())
print('FIT_SCRIPT =', DRIVE_PROJECT_ROOT / FIT_REL)


## Controlled Comparison Setup

**Why this section exists**
- The first analysis should match the Stage 7 subset exactly.

**Approach**
- Compare Stage 7 RGB against the best pose `6B` runs on `squat`, `pull_up`, and `push_up`.

**How to interpret the result**
- This isolates representation choice before introducing stronger RGB or multimodal variants.


In [ ]:
TARGET_CONFIGS = [
    {
        'exercise': 'squat',
        'pose_run': 'squat_tcn_l1_channels96',
        'rgb_run': 'rgb_count_tcn_squat_seq256',
    },
    {
        'exercise': 'pull_up',
        'pose_run': 'pose_count_tcn_pull_up_seq192',
        'rgb_run': 'rgb_count_tcn_pull_up_seq192',
    },
    {
        'exercise': 'push_up',
        'pose_run': 'pose_count_tcn_push_up_seq128',
        'rgb_run': 'rgb_count_tcn_push_up_seq128',
    },
]

TARGET_CONFIGS


## Run Representation-Fit Analysis

**Why this section exists**
- This produces per-video pose-vs-RGB comparisons for each exercise.

**Approach**
- Run the analysis script once per exercise.
- Write one CSV and one JSON summary per exercise beside the RGB run artifacts.

**How to interpret the result**
- The output files contain both the row-level evidence and the compact exercise summary.


In [ ]:
import subprocess
import pandas as pd

analysis_failures = []
for cfg in TARGET_CONFIGS:
    pose_pred = TRAINING_DIR / cfg['pose_run'] / 'predictions.csv'
    rgb_pred = TRAINING_DIR / cfg['rgb_run'] / 'predictions.csv'
    out_json = TRAINING_DIR / cfg['rgb_run'] / 'representation_fit_summary.json'
    out_csv = TRAINING_DIR / cfg['rgb_run'] / 'representation_fit_rows.csv'
    cmd = [
        'python', str(DRIVE_PROJECT_ROOT / FIT_REL),
        '--pose-summary-csv', str(POSE_SUMMARY),
        '--rgb-summary-csv', str(RGB_SUMMARY),
        '--pose-predictions-csv', str(pose_pred),
        '--rgb-predictions-csv', str(rgb_pred),
        '--exercise', cfg['exercise'],
        '--output-json', str(out_json),
        '--output-csv', str(out_csv),
        '--rgb-label', 'rgb_stage7',
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        analysis_failures.append({
            'exercise': cfg['exercise'],
            'returncode': exc.returncode,
        })

if analysis_failures:
    display(pd.DataFrame(analysis_failures))
else:
    print('All representation-fit analyses completed.')


## Exercise-Level Summary Review

**Why this section exists**
- This gives the compact answer: on which exercises does RGB win, and under what pose-quality conditions?

**Approach**
- Load the per-exercise JSON summaries.
- Display the top-line stats and the bucketed pose-quality breakdown.

**How to interpret the result**
- If RGB wins concentrate in low-visibility or low-confidence buckets, then RGB is compensating for pose weakness.


In [ ]:
import json
import pandas as pd

summary_rows = []
bucket_rows = []
missing_summary_paths = []
for cfg in TARGET_CONFIGS:
    summary_path = TRAINING_DIR / cfg['rgb_run'] / 'representation_fit_summary.json'
    if not summary_path.exists():
        missing_summary_paths.append(str(summary_path))
        continue
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    summary_rows.append({
        'exercise': cfg['exercise'],
        'rows': summary['rows'],
        'rgb_better_rows': summary['rgb_better_rows'],
        'pose_better_rows': summary['pose_better_rows'],
        'ties': summary['ties'],
        'mean_pose_abs_error': summary['mean_pose_abs_error'],
        'mean_rgb_abs_error': summary['mean_rgb_abs_error'],
        'mean_delta_rgb_minus_pose_abs_error': summary['mean_delta_rgb_minus_pose_abs_error'],
    })
    by_bucket = summary.get('by_pose_quality_bucket', {})
    for visibility, conf_map in by_bucket.items():
        for confidence, stats in conf_map.items():
            bucket_rows.append({
                'exercise': cfg['exercise'],
                'pose_visibility_bucket': visibility,
                'pose_conf_bucket': confidence,
                'rows': stats['rows'],
                'rgb_better_rows': stats['rgb_better_rows'],
                'pose_better_rows': stats['pose_better_rows'],
                'mean_pose_abs_error': stats['mean_pose_abs_error'],
                'mean_rgb_abs_error': stats['mean_rgb_abs_error'],
                'mean_delta_rgb_minus_pose_abs_error': stats['mean_delta_rgb_minus_pose_abs_error'],
            })

if missing_summary_paths:
    print('Missing representation-fit summaries:')
    for path in missing_summary_paths:
        print(' -', path)

summary_df = pd.DataFrame(summary_rows)
if summary_df.empty:
    print('No representation-fit summaries found yet.')
else:
    display(summary_df.sort_values('exercise'))

bucket_df = pd.DataFrame(bucket_rows)
if bucket_df.empty:
    print('No pose-quality bucket rows were produced for the current runs.')
else:
    display(bucket_df.sort_values(['exercise', 'pose_visibility_bucket', 'pose_conf_bucket']))


## Case Review Tables

**Why this section exists**
- The aggregate summary is useful, but the real decision often depends on a few concrete videos.

**Approach**
- Load the row-level CSVs.
- Show the strongest RGB wins and strongest pose wins per exercise.

**How to interpret the result**
- Use these rows to inspect whether the winning representation matches the visible data conditions in the raw videos.


In [ ]:
import pandas as pd

missing_rows_paths = []
for cfg in TARGET_CONFIGS:
    rows_path = TRAINING_DIR / cfg['rgb_run'] / 'representation_fit_rows.csv'
    if not rows_path.exists():
        missing_rows_paths.append(str(rows_path))
        continue
    df = pd.read_csv(rows_path)
    print(f'\nExercise: {cfg["exercise"]}')
    display(df.sort_values('delta_rgb_minus_pose_abs_error').head(10))
    display(df.sort_values('delta_rgb_minus_pose_abs_error', ascending=False).head(10))

if missing_rows_paths:
    print('\nMissing representation-fit row CSVs:')
    for path in missing_rows_paths:
        print(' -', path)
